## Math Background: 

For mathematical definitions and knowledge you can refer to this [notebook](mathematical_background.ipynb).

## Homomorphic encryption scheme in a nutshell

The **Cheon-Kim-Kim-Song** (CKKS) scheme is an **homomorphic encryption (HE)** scheme which allows the computation of some functions directly on encrypted data.

In particular, HE allows the computation of additions and multiplication, from which more sophisticated functionality can be built:

- between ciphertexts and ciphertexts;
- between ciphertexts and plaintexts.

The result of the operations, once decrypted, is the same as if it were applied on the corresponding plaintexts.

### Typical encryption:
$6.5 + 3.5 \neq 18.3$

E ⬇️ &nbsp; E ⬇️ &nbsp; D ⬆️

$18.5 + 11.3 = 19.8$

### Homomorphic encryption:
$6.5 + 3.5 \neq 10.0$

E ⬇️ &nbsp; E ⬇️ &nbsp; D ⬆️

$11.7 + 3.6 = 15.3$

An example would be hospitals encrypt patient data (e.g., blood pressure readings) using CKKS. A central server computes averages or other statistics directly on the encrypted data. The decrypted results reveal only aggregate insights, protecting individual privacy.

## Steps in CKKS

* **Encoding and Decoding:**
    * Given a vector of complex numbers or real numbers, encoding is performed to map it into a polynomial form, which is an element of a cyclotomic ring.
    * This process enables encryption and efficient homomorphic operations.
    * The advantage of this approach is that it allows CKKS to work with real numbers while leveraging the structure of integer polynomial rings.
    * Decoding transforms a polynomial $m \in \mathbb{R}$, output the original vector (complex or real numbers).

- **Encryption and Decryption:** 
    * Converts the encoded polynomial into ciphertext (encryption) and retrieves the plaintext polynomial from ciphertext (decryption).
      
* **Multiplication and Relinearization:**
    * Performs homomorphic multiplication of ciphertexts and relinearizes the result to reduce the size of the ciphertext and manage computational overhead.
- **Rescaling:**
    * Adjusts the scale of ciphertexts after operations to maintain precision and enable further operations without overflow issues.



**Frequently Asked Questions**:
1. [Vector Shorter than Number of Slots (N/2)?](https://stackoverflow.com/questions/62124656/the-form-of-plaintext-in-real-number-ckks-encoding-microsoft-seal)


## Plaintext Encoding and Decoding


### Encoding

For encoding, we aim to project a complex vector into the subspace:

$$
H = \{(z_j)_{j \in \mathbb{Z}^*_M} : z_j = z_{-j}\}.
$$

This is the $\pi^{-1}$ projection discussed in the original paper, where half of the entries are conjugates of the other half. We know that:

$$
\sigma(\mathcal{R}) \subseteq \mathbb{H} = \{ z \in \mathbb{C}^N : z_j = z_{-j} \}.
$$

The process involves the following steps:

1. **Projection using $\pi^{-1}$:**  
   The scaled vector is then projected using $\pi^{-1}$. This ensures that the resulting vector lies in $\mathbb{H}$.
   


In [320]:
# code inspired from https://colab.research.google.com/drive/1cdue90Fg_EB5cxxTYcv2_8_XxQnpnVWg?usp=sharing#scrollTo=qsDuOVro1H15

import numpy as np
from numpy.polynomial import Polynomial

In [321]:
def pi_inverse(z: np.array) -> np.array:
    """Expands a vector of C^{N/2} by expanding it with its
    complex conjugate."""
    
    z_conjugate = z[::-1]
    z_conjugate = [np.conjugate(x) for x in z_conjugate]
    return np.concatenate([z, z_conjugate])

# TOY EXAMPLE FROM https://eprint.iacr.org/2016/421.pdf section 3.2

M = 8 # parameter from the paper
delta = 64 # scaling factor 
xi = np.exp(2 * np.pi * 1j / M) # Mth root of unity which will be used as a basis for our computation
z = np.array([3 + 4j, 2 - 1j]) # vector to encode
pi_z = pi_inverse(z) # pi^-1
pi_z

array([3.+4.j, 2.-1.j, 2.+1.j, 3.-4.j])

2. **Multiplication with $\Delta$:**  
   We start by multiplying the vector with $\Delta$ to scale it appropriately.

In [322]:
scaled_pi_z = delta * pi_z
scaled_pi_z # scaled vector

array([192.+256.j, 128. -64.j, 128. +64.j, 192.-256.j])

3. **Coordinate-Wise Random Rounding:**  
   To project onto $\sigma(\mathcal{R})$, we use a technique called *coordinate-wise random rounding*. This is a method to discretize complex numbers in each coordinate independently while preserving certain statistical properties. For details, refer to [this paper](https://web.eecs.umich.edu/~cpeikert/pubs/toolkit.pdf).

    
   A. **Orthogonal $\mathbb{Z}$-Basis:**
   - The ring $\mathcal{R}$ has an orthogonal $\mathbb{Z}$-basis $\{1, X, \ldots, X^{N-1}\}$.
   - Since $\sigma$ is an isomorphism, $\sigma(\mathcal{R})$ has a corresponding orthogonal $\mathbb{Z}$-basis:
     $$
     \beta = \{b_1, b_2, \ldots, b_N\} = \{\sigma(1), \sigma(X), \ldots, \sigma(X^{N-1})\}.
     $$

In [323]:
def vandermonde(xi: np.complex128, M: int) -> np.array:
    """Computes the Vandermonde matrix from a m-th root of unity."""
    
    N = M // 2
    matrix = []
    # We will generate each row of the matrix
    for i in range(N):
        # For each row we select a different root
        root = xi ** (2 * i + 1)
        row = []

        # Then we store its powers
        for j in range(N):
            row.append(root ** j)
        matrix.append(row)
    return matrix

def create_sigma_R_basis(xi, M):
    """Creates the basis (sigma(1), sigma(X), ..., sigma(X** N-1))."""
    return np.array(vandermonde(xi, M)).T
    
sigma_R_basis = create_sigma_R_basis(xi, M)

We can now have a look at the basis $\{\sigma(1), \sigma(X), \ldots, \sigma(X^{N-1})\}$.

In [324]:
sigma_R_basis

array([[ 1.00000000e+00+0.j        ,  1.00000000e+00+0.j        ,
         1.00000000e+00+0.j        ,  1.00000000e+00+0.j        ],
       [ 7.07106781e-01+0.70710678j, -7.07106781e-01+0.70710678j,
        -7.07106781e-01-0.70710678j,  7.07106781e-01-0.70710678j],
       [ 1.79380389e-16+1.j        , -4.67705010e-16-1.j        ,
         7.70954637e-16+1.j        , -1.11479041e-15-1.j        ],
       [-7.07106781e-01+0.70710678j,  7.07106781e-01+0.70710678j,
         7.07106781e-01-0.70710678j, -7.07106781e-01-0.70710678j]])

B. **Projection in the Basis:**
- For any $z \in \mathbb{H}$, we can write:
     $$
     z = \sum_{i=1}^N z_i b_i,
     $$
     where $z_i$ are the coordinates of $z$ in the basis $\beta$. These coordinates are computed as:
     $$
     z_i = \frac{\langle z, b_i \rangle}{\|b_i\|^2} \in \mathbb{R}.
     $$
- Here, $\langle x, y \rangle$ is the **Hermitian inner product**:

    $$
    \langle x, y \rangle = \sum_{k=1}^N x_k \overline{y_k},
    $$
    where $\overline{y_k}$ denotes the complex conjugate of $y_k$.

In [325]:
def compute_basis_coordinates(z, sigma_R_basis):
    """Computes the coordinates of a vector with respect to the orthogonal lattice basis."""
    output = np.array([np.real(np.vdot(z, b) / np.vdot(b,b)) for b in sigma_R_basis])
    return output

coordinates = compute_basis_coordinates(scaled_pi_z, sigma_R_basis)

C. **Rounding to Integers:**
   - Since the coordinates $z_i$ are real numbers (not integers), we apply **coordinate-wise random rounding**:
     - Each $z_i$ is rounded to either $\lfloor z_i \rfloor$ or $\lfloor z_i \rfloor + 1$.
     - The probability of rounding to $\lfloor z_i \rfloor + 1$ is proportional to the fractional part of $z_i$.

In [326]:
def round_coordinates(coordinates):
    """Gives the integral rest."""
    coordinates = coordinates - np.floor(coordinates)
    return coordinates

def coordinate_wise_random_rounding(coordinates):
    """Rounds coordinates randonmly."""
    r = round_coordinates(coordinates)
    f = np.array([np.random.choice([c, c-1], 1, p=[1-c, c]) for c in r]).reshape(-1)
    
    rounded_coordinates = coordinates - f
    rounded_coordinates = [int(coeff) for coeff in rounded_coordinates]
    return rounded_coordinates

rounded_coordinates = coordinate_wise_random_rounding(coordinates)
rounded_coordinates

[160, 90, 160, 45]

D. **Reconstruction of $z$:**
   - Once we have the rounded coordinates $\{z_i\}$, we reconstruct $z$ using the basis $\beta$:
     $$
     z = \sum_{i=1}^N z_i b_i.
     $$


In [327]:
y = np.matmul(sigma_R_basis.T, rounded_coordinates)
y

array([191.81980515+255.45941546j, 128.18019485 -64.54058454j,
       128.18019485 +64.54058454j, 191.81980515-255.45941546j])

4. **Applying $\sigma^{-1}$:**  
Finally, we apply $\sigma^{-1}$, as described below. This step outputs an element in $\mathbb{R}$, completing the encoding process.

Find a polynomial $m(X) = \sum_{i=0}^{N-1} \alpha_i X^i \in \mathbb{C}[X]/(X^N + 1)$, given a vector $z \in \mathbb{C}^N$, such that $\sigma(m) = (m(\xi), m(\xi^3), \ldots, m(\xi^{2N-1})) = (z_1, \ldots, z_N)$ where $\xi_M$, the $M$-th root of unity: $\xi_M = e^{2i\pi/M}$. So we get:
$\sum_{j=0}^{N-1} \alpha_j (\xi^{2i-1})^j = z_i, \; i = 1, \ldots, N.$

This can be viewed as a linear equation:

$A\alpha = z$, with $A$ the Vandermonde matrix of the $(\xi^{2i-1})_{i=1,\ldots,N}$, $\alpha$ the vector of the polynomial coefficients, and $z$ the vector we want to encode.

$$
\begin{bmatrix}
1 & \xi_1 & \xi_1^2 & \xi_1^3 \\
1 & \xi_3 & \xi_3^2 & \xi_3^3 \\
1 & \xi_5 & \xi_5^2 & \xi_5^3 \\
1 & \xi_7 & \xi_7^2 & \xi_7^3
\end{bmatrix}
\cdot
\begin{bmatrix}
a_1 \\
a_2 \\
a_3 \\
a_4
\end{bmatrix}
=\begin{bmatrix}
z_1 \\
z_2 \\
z_3 \\
z_4
\end{bmatrix}
$$

where $\xi_1 = \xi_M, \xi_3 = (\xi_M)^3, \xi_5 = (\xi_M)^5, \ldots$.



In [328]:
def sigma_inverse(b: np.array, M) -> Polynomial:
    """Encodes the vector b in a polynomial using an M-th root of unity."""

    # First we create the Vandermonde matrix
    A = vandermonde(xi, M)

    # Then we solve the system
    coeffs = np.linalg.solve(A, b)

    # Finally we output the polynomial
    p = Polynomial(coeffs)
    
    # We round it afterwards due to numerical imprecision
    coef = np.round(np.real(p.coef)).astype(int)
    return Polynomial(coef)

encoded = sigma_inverse(y, M)
encoded

Polynomial([160.,  90., 160.,  45.], domain=[-1.,  1.], window=[-1.,  1.], symbol='x')

Therefore, we have that:

$\alpha = A^{-1}z$, and that $\sigma^{-1}(z) = \sum_{i=0}^{N-1} \alpha_i X^i \in \mathbb{C}[X]/(X^N + 1)$. 

### Decoding

For decoding, we work with a cyclotomic polynomial ring $\mathbb{Z}[X]/(X^N + 1)$ and aim to decode it to a vector $z \in \mathbb{C}^N$ such that:
$$
z = \pi \circ \sigma(\Delta^{-1} * m) \in \mathbb{C}^{N/2}.
$$

The decoding process involves the following steps:


1. **Scaling by $1 / \Delta$:**  
   Multiply the polynomial $m$ by the inverse of $\Delta$, i.e., $1 / \Delta$. This rescales the polynomial back to the correct domain.


In [329]:
rescaled_p = encoded / delta

2. **Applying $\sigma$:**  
   Next, apply $\sigma$, which maps the polynomial from $\mathbb{Z}[X]/(X^N + 1)$ to a complex vector in $\mathbb{C}^N$. This is explained in detail below.

To decode a polynomial $m(X)$ into a vector $z$, we evaluate on certain values, which will be the roots of $\Phi(M) = X^N + 1$, where the $N$ roots are $\xi, \xi^3, \ldots, \xi^{2N-1}$. 


In [330]:
def sigmaf(p: Polynomial, M) -> np.array:
    """Decodes a polynomial by applying it to the M-th roots of unity."""

    outputs = []
    N = M // 2

    # We simply apply the polynomial on the roots
    for i in range(N):
        root = xi ** (2 * i + 1)
        output = p(root)
        outputs.append(output)
    return np.array(outputs)

z_vector = sigmaf(rescaled_p, M)

3. **Projection to $\mathbb{C}^{N/2}$:**  
   The final step is to project the vector onto $\mathbb{C}^{N/2}$. Since the structure ensures that the second half of the array is the conjugate of the first half, this projection simply involves returning the first half of the array.

In [331]:
def pi(z: np.array, M) -> np.array:
    """Projects a vector of H into C^{N/2}."""
    
    N = M // 4
    return z[:N]

decoded = pi(z_vector, M)
decoded

array([2.99718446+3.99155337j, 2.00281554-1.00844663j])

If we put everything together we have:

In [332]:
def sigma_R_discretization(z, sigma_R_basis):
    """Projects a vector on the lattice using coordinate wise random rounding."""
    coordinates = compute_basis_coordinates(z, sigma_R_basis)
    
    rounded_coordinates = coordinate_wise_random_rounding(coordinates)
    y = np.matmul(sigma_R_basis.T, rounded_coordinates)
    return y

def decode(p: Polynomial, scale, M) -> np.array:
    """Decodes a polynomial by removing the scale, 
    evaluating on the roots, and project it on C^(N/2)."""
    rescaled_p = p / scale
    z = sigmaf(rescaled_p, M)
    pi_z = pi(z, M)
    return pi_z

def encode(z: np.array, scale, M) -> Polynomial:
    """Encodes a vector by expanding it first to H,
    scale it, project it on the lattice of sigma(R), and performs
    sigma inverse.
    """
    sigma_R_basis = create_sigma_R_basis(xi, M)
    pi_z = pi_inverse(z)
    scaled_pi_z = scale * pi_z
    rounded_scale_pi_zi = sigma_R_discretization(scaled_pi_z, sigma_R_basis)
    p = sigma_inverse(rounded_scale_pi_zi, M)
    
    # We round it afterwards due to numerical imprecision
    coef = np.round(np.real(p.coef)).astype(int)
    p = Polynomial(coef)
    return p


# Another example with larger Delta

delta = 2**16
encoded_pol = encode(z, delta, M)
decoded = decode(encoded_pol, delta, M)

print(f"Original vector: ", z)
print(f"Encoded polynoial: ", encoded_pol) 
print(f"Vector after encoding and decoding: ", decoded)

Original vector:  [3.+4.j 2.-1.j]
Encoded polynoial:  163840.0 + 92682.0·x + 163840.0·x² + 46341.0·x³
Vector after encoding and decoding:  [3.00000054+4.00000162j 1.99999946-0.99999838j]


We will use this encoded poynomial $163840.0 + 92682.0·x + 163840.0·x² + 46341.0·x³$ as an example for encryption and decryption later.

### Alternative Encoding with FFT

> We can compute the same CKKS encoding and decoding maps with FFT because the roots in the Vandermonde matrix are powers of unity.

In the walkthrough above, we used:
- basis projections with Hermitian inner products;
- reconstruction in the $\sigma(\mathcal{R})$ basis;
- a linear solve for $\sigma^{-1}$;
- direct evaluation for $\sigma$.

For small toy parameters this is already easy to understand. For larger $N$, the same maps can be written with FFT, which makes the implementation much faster while keeping the same mathematics.

In [333]:
def compute_basis_coordinates_fft(z, M):
    """Computes the basis coordinates with an FFT formula."""

    N = M // 2
    xi = np.exp(2 * np.pi * 1j / M)
    phase = xi ** (-np.arange(N))
    coordinates = phase * np.fft.fft(np.asarray(z, dtype=np.complex128)) / N
    return np.real_if_close(coordinates).real

def sigma_inverse_fft(b, M) -> Polynomial:
    """Encodes the vector b in a polynomial using FFT."""

    N = M // 2
    xi = np.exp(2 * np.pi * 1j / M)
    phase = xi ** (-np.arange(N))

    # This is the same sigma inverse as before, but written as a Fourier transform.
    coeffs = phase * np.fft.fft(np.asarray(b, dtype=np.complex128)) / N
    coeffs = np.round(np.real(coeffs)).astype(int)
    return Polynomial(coeffs)

def sigmaf_fft(p: Polynomial, M) -> np.array:
    """Decodes a polynomial by applying an FFT-based sigma map."""

    N = M // 2
    xi = np.exp(2 * np.pi * 1j / M)
    phase = xi ** np.arange(N)

    coeffs = np.zeros(N, dtype=np.complex128)
    raw_coeffs = np.asarray(p.coef, dtype=np.complex128)
    coeffs[:min(len(raw_coeffs), N)] = raw_coeffs[:N]

    # Multiplication by the phase turns the odd-root evaluation into an FFT.
    phased_coeffs = coeffs * phase
    return np.fft.ifft(N * phased_coeffs)

def encode_fft(z: np.array, scale, M, rounded_coordinates=None) -> Polynomial:
    """Encodes a vector with the same CKKS steps, but using FFT."""

    pi_z = pi_inverse(z)
    scaled_pi_z = scale * pi_z
    coordinates = compute_basis_coordinates_fft(scaled_pi_z, M)

    if rounded_coordinates is None:
        rounded_coordinates = coordinate_wise_random_rounding(coordinates)

    y = sigmaf_fft(Polynomial(rounded_coordinates), M)
    return sigma_inverse_fft(y, M)

def decode_fft(p: Polynomial, scale, M) -> np.array:
    """Decodes a polynomial with the FFT version of sigma."""

    rescaled_p = p / scale
    z = sigmaf_fft(rescaled_p, M)
    return pi(z, M)

#### Step-by-Step Idea

> We follow the same four steps as before, but we replace the Vandermonde matrix and the linear solve with FFT formulas.

1. Expand the vector with $\pi^{-1}$ and scale it by $\Delta$.
2. Compute the basis coordinates with an FFT instead of Hermitian products against every basis vector.
3. Reconstruct the lattice point and recover the polynomial coefficients with an FFT-based version of $\sigma^{-1}$.
4. Decode the polynomial with an FFT-based version of $\sigma$.

The mathematics is the same. The difference is only that FFT gives a much faster way to compute the same maps when $N$ becomes large.

In [334]:
# We reuse the same small toy example from the earlier walkthrough.
# The only change is that the linear algebra is now done with FFT formulas.


fft_delta = 64
fft_z = np.array([3 + 4j, 2 - 1j])
fft_pi_z = pi_inverse(fft_z)
fft_scaled_pi_z = fft_delta * fft_pi_z

# Step 1: compute the basis coordinates with the FFT formula.
fft_coordinates = compute_basis_coordinates_fft(fft_scaled_pi_z, M)

# Step 2: reuse the same rounded coordinates as in the manual walkthrough above.
fft_rounded_coordinates = [160, 91, 160, 45]

# Step 3: reconstruct sigma(R) and recover the polynomial coefficients with FFT.
fft_y = sigmaf_fft(Polynomial(fft_rounded_coordinates), M)
fft_encoded = sigma_inverse_fft(fft_y, M)

# Step 4: decode again with the FFT version of sigma.
fft_decoded = decode_fft(fft_encoded, fft_delta, M)

print(f"Original vector: {fft_z}")
print(f"Coordinates from FFT: {np.round(fft_coordinates, 6)}")
print(f"Rounded coordinates: {fft_rounded_coordinates}")
print(f"Encoded polynomial with FFT: {fft_encoded}")
print(f"Vector after FFT encoding and decoding: {fft_decoded}")

Original vector: [3.+4.j 2.-1.j]
Coordinates from FFT: [160.        90.509668 160.        45.254834]
Rounded coordinates: [160, 91, 160, 45]
Encoded polynomial with FFT: 160.0 + 91.0·x + 160.0·x² + 45.0·x³
Vector after FFT encoding and decoding: [3.008233+4.00260191j 1.991767-0.99739809j]


In [335]:
# code inspired from https://github.com/AI-Tech-Research-Lab/Introduction-to-BFV-HE-ML/blob/main/BFV_theory/BFV_theory.ipynb

import random

def polynomial_from_dict(coefficients_by_power):
    """Builds a polynomial as a list of exact integer coefficients."""

    # We first create enough space for the largest power that appears.
    max_power = max(coefficients_by_power.keys()) if coefficients_by_power else 0
    coefficients = [0] * (max_power + 1)

    # Then we place each coefficient in the position that matches its power.
    for power, coefficient in coefficients_by_power.items():
        coefficients[power] = int(round(coefficient))
    return coefficients

def polynomial_to_coefficients(polynomial: Polynomial):
    """Converts a Polynomial into exact integer coefficients."""

    # We round afterwards only to remove tiny numerical imprecision.
    return [int(round(coefficient)) for coefficient in polynomial.coef]

def trim_coefficients(coefficients):
    """Removes trailing zero coefficients."""
    trimmed = [int(round(coefficient)) for coefficient in coefficients]
    while len(trimmed) > 1 and trimmed[-1] == 0:
        trimmed.pop()
    return trimmed

def pad_coefficients(coefficients, size):
    """Extends a coefficient list with zeros up to the requested size."""
    padded = [int(round(coefficient)) for coefficient in coefficients]
    if len(padded) >= size:
        return padded[:size]
    return padded + [0] * (size - len(padded))

def pr(polynomial):
    """Pretty-print a polynomial stored as exact coefficients."""
    coefficients = trim_coefficients(polynomial)
    terms = []

    for power, coefficient in enumerate(coefficients):
        if coefficient == 0:
            continue

        sign = "-" if coefficient < 0 else "+"
        absolute_coefficient = abs(coefficient)

        if power == 0:
            term = f"{absolute_coefficient}"
        elif power == 1:
            term = f"{absolute_coefficient}X" if absolute_coefficient != 1 else "X"
        else:
            term = f"{absolute_coefficient}X**{power}" if absolute_coefficient != 1 else f"X**{power}"

        terms.append((sign, term))

    if not terms:
        return "0"

    first_sign, first_term = terms[0]
    result = first_term if first_sign == "+" else f"- {first_term}"
    for sign, term in terms[1:]:
        result += f" {sign} {term}"
    return result

def coeffs_to_polynomial_string(coefficients):
    return pr(coefficients)

def center_coefficients(coefficients, q):
    """Maps coefficients from Z_q back to the centered interval."""
    centered = []
    for coefficient in coefficients:
        reduced = coefficient % q
        if reduced > q // 2:
            reduced -= q
        centered.append(reduced)
    return centered

def add_plain(left, right):
    """Adds two coefficient lists without applying the modulus yet."""
    size = max(len(left), len(right))
    left = pad_coefficients(left, size)
    right = pad_coefficients(right, size)
    return [left_coefficient + right_coefficient for left_coefficient, right_coefficient in zip(left, right)]

def sub_plain(left, right):
    """Subtracts two coefficient lists without applying the modulus yet."""
    size = max(len(left), len(right))
    left = pad_coefficients(left, size)
    right = pad_coefficients(right, size)
    return [left_coefficient - right_coefficient for left_coefficient, right_coefficient in zip(left, right)]

def multiply_plain(left, right):
    """Multiplies two polynomials before reducing modulo x^N + 1."""
    output = [0] * (len(left) + len(right) - 1)

    # We multiply every term of the first polynomial with every term of the second.
    for left_power, left_coefficient in enumerate(left):
        if left_coefficient == 0:
            continue
        for right_power, right_coefficient in enumerate(right):
            if right_coefficient == 0:
                continue
            output[left_power + right_power] += left_coefficient * right_coefficient
    return output

def mod_on_coefficients(polynomial, modulo):
    """Applies the coefficient modulus exactly."""
    return [coefficient % modulo for coefficient in polynomial]

def reduce_polynomial(polynomial, polynomial_modulous, q):
    """Reduces modulo x^N + 1 and then modulo q."""
    modulus_coefficients = polynomial_modulous
    N = len(modulus_coefficients) - 1
    reduced = [0] * N

    # Every time a term passes degree N - 1, it wraps around with a sign flip.
    for index, coefficient in enumerate(polynomial):
        quotient, remainder = divmod(index, N)
        sign = -1 if quotient % 2 else 1
        reduced[remainder] += sign * coefficient

    return [coefficient % q for coefficient in reduced]

def add_poly_exact(left, right, q):
    """Adds two polynomials in Z_q[X]/(x^N + 1)."""
    return mod_on_coefficients(add_plain(left, right), q)

def negate_poly_exact(polynomial, q):
    """Negates a polynomial coefficient-wise modulo q."""
    return [(-coefficient) % q for coefficient in polynomial]

def mul_poly_exact(left, right, q):
    """Multiplies two polynomials in Z_q[X]/(x^N + 1)."""
    N = max(len(left), len(right))
    left = pad_coefficients(left, N)
    right = pad_coefficients(right, N)
    output = [0] * N

    # Terms of degree >= N wrap back because x^N = -1 in the quotient ring.
    for left_power, left_coefficient in enumerate(left):
        if left_coefficient == 0:
            continue
        for right_power, right_coefficient in enumerate(right):
            if right_coefficient == 0:
                continue
            index = left_power + right_power
            term = left_coefficient * right_coefficient
            if index >= N:
                index -= N
                term = -term
            output[index] += term

    return [coefficient % q for coefficient in output]

def find_negative_coefficients(polynomial):
    """Returns the positions of negative coefficients before reducing modulo q."""
    return [index for index, coefficient in enumerate(polynomial) if coefficient < 0]

def to_numpy_polynomial(coefficients):
    """Converts exact coefficients back to a Polynomial for decoding."""
    return Polynomial(np.array(coefficients, dtype=np.float64))

def secret_key_pol(N, h):
    """Generates the secret key with coefficients in {-1, 0, 1}."""
    secret = [0] * N

    # We choose h positions that will contain either 1 or -1.
    non_zero_indices = random.sample(range(N), h)
    for index in non_zero_indices:
        secret[index] = random.choice([-1, 1])
    return secret

def create_error_pol(N, Sigma, mu):
    """Generates an error polynomial with exact integer coefficients."""
    return [int(round(value)) for value in np.random.normal(mu, Sigma, N)]

def sample_pol(N, modulous):
    """Samples a polynomial with coefficients in Z_q."""
    return [int(np.random.randint(0, modulous)) for _ in range(N)]

def sample_ternary_pol(N):
    """Samples a polynomial with coefficients in {-1, 0, 1}."""
    return [int(np.random.randint(-1, 2)) for _ in range(N)]

## Ciphertext Encryption and Decryption

The schemes like BFV or also CKKS are based on a hard computation problem called Ring Learning With Errors.

### Coefficients

First, the coefficients of the polynomials in RLWE are whole numbers, and they are always taken **modulo** some fixed number $q$. This means that each coefficient is the **remainder** when divided by $q$, effectively keeping them within a limited range.

#### Example:
Let’s take the modulus $q = 12$. You can think of this like a **clock**, where each hour corresponds to an element in the range $\{0, 1, 2, \dots, 11\}$. On a clock, if you add $10 + 5$, the result isn’t $15$, instead it wraps around to $3$.

Similarly, in modular arithmetic, coefficients work the same way. For example:

$$
10 + 5 \mod 12 = 3
$$

---

#### Ring of Polynomials
Now assume we are working with polynomials instead of just numbers. A polynomial like:

$$
f(x) = 10x + 5
$$

is in the "ring of polynomials," where each coefficient $10$ and $5$ is taken **modulo $q$**. So, if $q = 12$, the polynomial remains valid and computations wrap around, just like on the clock.

### Polynomial Modulus

The second important aspect is that the **polynomials themselves are also reduced modulo a special polynomial**. This special polynomial is called the **polynomial modulus**. Every polynomial used in the scheme is divided by this polynomial modulus, and only the remainder is kept.

In homomorphic encryption (HE) schemes, the polynomial modulus is commonly chosen in the form:

$$
f(x) = x^N + 1
$$

where $N$ is a power of $2$. For example, if we take $N = 8$, the polynomial modulus becomes:

$$
f(x) = x^8 + 1
$$

---

### Key Characteristics of Polynomials in the Scheme

The polynomials used in the scheme have two important properties:

1. **Coefficients are reduced modulo $q$**:
   Each coefficient in the polynomial is reduced modulo $q$, creating a ring structure where coefficients belong to the range $\{0, 1, 2, \dots, q-1\}$.

2. **Maximum degree is $N-1$**:
   The polynomials are reduced modulo the polynomial $f(x) = x^N + 1$, meaning their degree can be at most $N-1$.

---

If $q = 23$ and $N = 8$, then the polynomials in the scheme are of the form:

$$
p(x) = a_0 + a_1x + a_2x^2 + \dots + a_7x^7
$$

where each coefficient $a_i$ satisfies $0 \leq a_i < 23$. This means there are 8 coefficients, each in the range from $0$ to $22$.

### Practical Example
Let's take 2 polynomials $a = 3x^{4}$ and $b = 4x^{5}$ with $N=8$ -> polynomial modulous $x^{8} + 1$ and $q=7$. When we multiple $a*b$, we get $12x^9$.

After each operation, we have to perform the division with the polynomial modulus and keep the remainder.

$$
12x^{9} \mod (x^8 + 1) = -12x
$$

Also, the result's coefficients have to be taken $\mod q$.

$[-12x]_q = 2x$

In [336]:
a = polynomial_from_dict({4: 3})
b = polynomial_from_dict({5: 4})
q = 7
polynomial_modulus = polynomial_from_dict({0: 1, 8: 1})

prod = multiply_plain(a, b)
final_result = reduce_polynomial(prod, polynomial_modulus, q)

print(f"A: {pr(a)}")
print(f"B: {pr(b)}")
print(f"Product of A and B: {pr(prod)}")
print(f"Polynomial modulus: {pr(polynomial_modulus)}")
print(f"----------------------")
print(f"Remainder of (A*B) / polynomial modulus: {pr(center_coefficients(final_result, q))}")
print(f"Apply also mod q: {pr(final_result)}")

A: 3X**4
B: 4X**5
Product of A and B: 12X**9
Polynomial modulus: 1 + X**8
----------------------
Remainder of (A*B) / polynomial modulus: 2X
Apply also mod q: 2X


### CKKS Encryption and Decryption

In CKKS, encryption takes a plaintext (typically a polynomial of integer numbers) and converts it into a ciphertext using a **public key** derived from a **private key**. The decryption process, which recovers the plaintext, is feasible only if you know the private key.

---

#### Ciphertext
The encryption of a plaintext produces a ciphertext, which is represented by **two or more polynomials** from the same ring. However, the ciphertext uses:
- The same **polynomial modulus** $f(x) = x^N + 1$.
- A much larger modulus $q$, known as the **ciphertext coefficient modulus**, to allow space for noise during homomorphic computations.


#### Example with Small Parameters
To simplify, let’s consider smaller, insecure parameters for an example:
- $N = 4$, so $f(x) = x^4 + 1$..
- $q = ^{60} - 1 $ (ciphertext modulus).

##### Example Plaintext:
A plaintext polynomial might be:

$$
p(x) = 163840 + 92682x + 163840x^2 + 46341^3
$$


##### Encryption:
Encryption transforms $p(x)$ into a ciphertext $(c_0(x), c_1(x))$, where both $c_0(x)$ and $c_1(x)$ are polynomials with coefficients reduced modulo $q = 2^{60} - 1 $.


In [337]:
q = 2**60 - 1 
N = 4
m = polynomial_from_dict({0: 163840, 1: 92682, 2: 163840, 3:46341})

polynomial_modulus = polynomial_from_dict({0: 1, N: 1})

print(f"Ciphertext coefficient modulo: {q}")
print(f"Polynomial modulus: {pr(polynomial_modulus)}")
print(f"Plaintext polynomial: {pr(m)}")

Ciphertext coefficient modulo: 1152921504606846975
Polynomial modulus: 1 + X**4
Plaintext polynomial: 163840 + 92682X + 163840X**2 + 46341X**3


#### Generation of Private Key $s$

In CKKS (and similar schemes), the private key $s$ is a randomly generated polynomial with certain constraints. Here’s a brief explanation:

1. **Parameter $h$**:
   - The parameter $h$ determines the number of non-zero coefficients in the secret key $s$.

2. **Structure of $s$**:
   - $s(x)$ is a polynomial in the ring $R = \mathbb{Z}[x]/(f(x))$, where $f(x) = x^N + 1$.
   - The coefficients of $s(x)$ are either $-1$, $0$, or $1$.

3. **Generation Process**:
   - Randomly select $h$ positions in the polynomial $s(x)$ to be non-zero.
   - Assign these positions a value of $1$ or $-1$ (randomly chosen).
   - The remaining coefficients are set to $0$.

In [338]:
h = 2
s = secret_key_pol(N, h) # secret key
print(f"Secret key: {pr(s)}")

Secret key: X**2 + X**3


#### Generation of Public Key $(a, b)$

In CKKS (and similar schemes), the public key is derived from the private key $s$ and includes two polynomials $(a, b)$. Here’s a brief explanation:

1. **Structure of the Public Key**:
   - The public key $(a, b)$ consists of two polynomials in the ring $R = \mathbb{Z}[x]/(f(x))$, where $f(x) = x^N + 1$.
   - The coefficients of these polynomials are reduced modulo a large number $q$ (ciphertext modulus).

2. **Generation Process**:
   - Randomly generate a polynomial $a(x)$ with coefficients modulo $q$.
   - Compute $b(x)$ as:
     $$
     b(x) = -a(x) \cdot s(x) + e(x) \pmod{q}
     $$
     where:
     - $s(x)$ is the private key.
     - $e(x)$ is a small error (noise) polynomial, with coefficients sampled from a discrete Gaussian or uniform distribution to ensure security.

3. **Result**:
   - The public key is $(a(x), b(x))$, where:
     - $a(x)$ is the random polynomial.
     - $b(x)$ is derived using the private key and noise.


In [339]:
mu, Sigma = 0, 3.2 # suggested values in the original paper
a = sample_pol(N, q)
e = create_error_pol(N, Sigma, mu)

b = add_poly_exact(negate_poly_exact(mul_poly_exact(a, s, q), q), e, q)

# public key
pk = (b, a)

print(f"a: {pr(a)}\n")
print(f"e: {pr(e)}\n")
print(f"Public key: {pr(pk[0])}, {pr(pk[1])}\n")

a: 372963635737465808 + 371145595790302917X + 519400488881227599X**2 + 1060911828415941056X**3

e: 5 - X + X**2 - 3X**3

Public key: 890546084671530521 + 427390812690321679X + 687948192678475249X**2 + 408812273079078247X**3, 372963635737465808 + 371145595790302917X + 519400488881227599X**2 + 1060911828415941056X**3



To perform the encryption we need three "small" polynomials:

- Two error polynomials ("small error" polynomials), extracted from a discrete Gaussian distribution (similarly to the one used in the public key);
- A "small" polynomial, $v$ which has coefficients drawn from $(-1, 0, 1)$, similar to $s$.

In [340]:
e1 = create_error_pol(N, Sigma, mu)
e2 = create_error_pol(N, Sigma, mu)
v = sample_ternary_pol(N)

print(f"e_1: {pr(e1)}\n")
print(f"e_2: {pr(e2)}\n")
print(f"v: {pr(v)}\n")

e_1: - 8 + 2X + 3X**2 - 2X**3

e_2: 7 - 5X - X**3

v: 1 - X - X**2



### Encryption in CKKS

In CKKS, the ciphertext is represented by two polynomials:

$$
\text{ct} = \left( \left[ pk[0] \cdot v + e_1 + m \right]_{\Phi_N, q}, \, \left[ pk[1] \cdot v + e_2 \right]_{\Phi_N, q} \right)
$$

Where:
- $m$: The plaintext polynomial
- $v$: A random polynomial for security.
- $e_1, e_2$: Small noise polynomials.
- $q$: Ciphertext modulus.
- $\Phi_N = x^N + 1$: The polynomial modulus.

In [341]:
ct0 = add_poly_exact(add_poly_exact(mul_poly_exact(pk[0], v, q), e1, q), m, q)
ct1 = add_poly_exact(mul_poly_exact(pk[1], v, q), e2, q)

ciphertext = (ct0, ct1)
print(f"Ciphertext: {pr(ct0)}, {pr(ct1)}")

Ciphertext: 834385045822400874 + 1098578505704809064X + 522932799923633867X**2 + 446394772317174633X**3, 800354448427787495 + 1059093788468778160X + 928212761960305849X**2 + 170365743744410539X**3


### Decryption in CKKS

The decryption process in CKKS is relatively straightforward. The ciphertext is represented as two polynomials $(c_0, c_1)$, and decryption uses the private key $s$.

Multiply the second term of the ciphertext $c_1$ with the private key $s$, and sum it with the first term $c_0$:
   $$
   \text{Dec} = c_0 + c_1 \cdot s \pmod{\Phi_N, q}
   $$

   Substituting the ciphertext structure, we get:
   $$
   \text{Dec} = \left[\Delta \cdot m - e_v - e_1 + e_2\cdot s\right]_{\Phi_N, q}
   $$

Inside this polynomial we have the scaled message summed to some noise. If the noise is not too big, we can recover the message.

To do that, we just try to make the modulo with the polynomial modulus, than to apply
to the coefficients of the resulting polynomial.

In [342]:
plaintext = add_poly_exact(mul_poly_exact(ciphertext[1], s, q), ciphertext[0], q)
plaintext_centered = center_coefficients(plaintext, q)
plaintext_polynomial = to_numpy_polynomial(plaintext_centered)

print(f"Plain starting message: {pr(m)}")
print(f"Final decryption result: {pr(plaintext_centered)}")

Plain starting message: 163840 + 92682X + 163840X**2 + 46341X**3
Final decryption result: 163840 + 92676X + 163848X**2 + 46338X**3


Then if we try to decode the decrypted result, we get:

In [343]:
delta = 2**16
decoded_vector = decode(plaintext_polynomial, delta, N*2)

print(f"Original vector: ", z)
print(f"Decoded vector: {decoded_vector}")

Original vector:  [3.+4.j 2.-1.j]
Decoded vector: [2.99996817+4.00002658j 2.00003183-1.00021756j]


Following the decoding process, we obtain a high-quality approximation of the original input vector.

The steps achieved thus far can be summarized as follows:

**Input Vector → Encoding → Encryption → Decryption → Decoding → Approximation of Input Vector**

We are ready to define some other helper functions to automatize the process of encryption and decryption.

In [344]:
def generate_keys(N, q, h, pol_modulus, mu=0, Sigma=3.2, a=None, e=None, seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    sk = secret_key_pol(N, h)
    a = sample_pol(N, q) if a is None else pad_coefficients(a, N)
    e = create_error_pol(N, Sigma, mu) if e is None else pad_coefficients(e, N)
    
    b = add_poly_exact(negate_poly_exact(mul_poly_exact(a, sk, q), q), e, q)
    pk = (b, a)

    return sk, pk

def encrypt_poly(m, N, pk, pol_modulus, q, mu=0, Sigma=3.2, e1=None, e2=None, v=None):
    message = pad_coefficients(m, N)
    e1 = create_error_pol(N, Sigma, mu) if e1 is None else pad_coefficients(e1, N)
    e2 = create_error_pol(N, Sigma, mu) if e2 is None else pad_coefficients(e2, N)
    v = sample_ternary_pol(N) if v is None else pad_coefficients(v, N)

    ct0 = add_poly_exact(add_poly_exact(mul_poly_exact(pk[0], v, q), e1, q), message, q)
    ct1 = add_poly_exact(mul_poly_exact(pk[1], v, q), e2, q)
    return ct0, ct1

def decrypt_poly(ct, sk, pol_modulus, q):
    plaintext = add_poly_exact(mul_poly_exact(ct[1], sk, q), ct[0], q)
    return plaintext

Now we will show a complete example with endoding/decoding and encryption/decryption for a vector of 2 real numbers.


In [345]:
# PARAMETERS
N = 4
h = 2
polynomial_modulus = polynomial_from_dict({0: 1, N: 1})
q = 2**60-1
delta = 2**40 # scaling factor 

# vector with 2 integers
z = np.array([4.0 + 0j, 3 + 0j])
print(f"Original vector: ", z)

Original vector:  [4.+0.j 3.+0.j]


In [346]:
# VECTOR ENCODING
encoded_pol = encode(z, delta, N*2) # encoded polynomial
encoded_coeffs = polynomial_to_coefficients(encoded_pol)
neg_coeff = find_negative_coefficients(encoded_coeffs) # store the positions with negative coefficients
encoded_pol_after_mod = mod_on_coefficients(encoded_coeffs, q) # to ensure that the coefficients are within Zq

print(f"Encoded polynomial: {pr(encoded_coeffs)}")
print(f"Encoded polynomial after mod q: {pr(encoded_pol_after_mod)}")

Encoded polynomial: 3848290697216 + 388736063997X - 388736063997X**3
Encoded polynomial after mod q: 3848290697216 + 388736063997X + 1152921115870782978X**3


As we can see when we encode our vector the coefficients must be modulo q, so we perform modulous operation on the coefficients. We can see that only the coefficient of $x^3$ changed because it was a negative number.

In [347]:
# GENERATE THE SECRET AND PUBLIC KEYS AND THEN ENCRYPT THE POLYNOMIAL
secret_key, public_key = generate_keys(N, q, h, polynomial_modulus)
encrypted = encrypt_poly(encoded_pol_after_mod, N, public_key, polynomial_modulus, q)
print(f"Ciphertext: {pr(encrypted[0])}, {pr(encrypted[1])}")

Ciphertext: 1028761594742123244 + 924076590622872146X + 812417569761707813X**2 + 71409951659192744X**3, 1110963771394058293 + 770459836548919144X + 841870176944175878X**2 + 966033935099596823X**3


In [348]:
# DECRYPT POLYNOMIAL
decrypted_coeffs = decrypt_poly(encrypted, secret_key, polynomial_modulus, q)

# We then change back the coefficients, where the mod operation changed them
coeffs = decrypted_coeffs[:] # get the coefficients of the decrypted polynomial
for x in neg_coeff: 
    coeffs[x] -= q # perform coefficient - q for the initial negative coefficients

decrypted_polynomial = to_numpy_polynomial(coeffs)
print(f"Decrypted polynomial: {pr(coeffs)}")

Decrypted polynomial: 3848290697214 + 388736064005X + 1152921504606846962X**2 - 388736063990X**3


Finally we decode the vector.

In [349]:
decoded = decode(decrypted_polynomial, delta, N*2) 

print(f"Original vector: ", z)
print(f"Vector after decoding: ", decoded)

Original vector:  [4.+0.j 3.+0.j]
Vector after decoding:  [4.+1048576.j 3.-1048576.j]


Again after the decoding process, we obtain a high-quality approximation of the original input vector.

## Homomorphic Operations

The core of homomorphic encryption are the operations that can be done on encrypted data, that includes addition on ciphertext and plaintext, ciphertext and ciphertext, as well as multiplication of ciphertext and plaintext, as well as ciphertext and ciphertext. Another example that will be shown is squaring a ciphertext.

## Addition

Addition in CKKS is rather straightforward, we have the following ciphertexts:
$$ c_1 = (c_{10}, c_{11})$$
$$ c_2 = (c_{20}, c_{21})$$

$$ c_{add} = c_1 + c_2 = (c_{10} + c_{20}, c_{11} + c_{21})$$

With addition, the resulting ciphertext contains both the original message terms and an accumulated error term. Since the noise terms are typically miniscule compared to the scaling factor, the accumulated error remains manageable.

We are going to show an example with adding 2 polynomials: 

In [350]:
N = 4
h = 2
polynomial_modulus = polynomial_from_dict({0: 1, N: 1})
q = 2**40-1
delta = 2**30 # scaling factor 

m1 = mod_on_coefficients(polynomial_from_dict({0: 163840, 1: 92682, 2: 123821, 3: 46341}), q)
m2 = mod_on_coefficients(polynomial_from_dict({0: 204800, 1: 103456, 2: 294721, 3: 57321}), q)

secret_key, public_key = generate_keys(N, q, h, polynomial_modulus)

print(f"First message: {pr(m1)}")
print(f"Second message: {pr(m2)}")
print(f"Expected sum: {pr(add_poly_exact(m1, m2, q))}")
print(f"-------------------------")

enc_m1 = encrypt_poly(m1, N, public_key, polynomial_modulus, q)
enc_m2 = encrypt_poly(m2, N, public_key, polynomial_modulus, q)

print(f"First ciphertext: {pr(enc_m1[0])}, {pr(enc_m1[1])}\n")
print(f"Second ciphertext: {pr(enc_m2[0])}, {pr(enc_m2[1])}\n")

First message: 163840 + 92682X + 123821X**2 + 46341X**3
Second message: 204800 + 103456X + 294721X**2 + 57321X**3
Expected sum: 368640 + 196138X + 418542X**2 + 103662X**3
-------------------------
First ciphertext: 174475171086 + 681164661343X + 9245796496X**2 + 369589790514X**3, 525377156415 + 467141146598X + 943724215530X**2 + 457895473929X**3

Second ciphertext: 106993529555 + 788157996876X + 797403860812X**2 + 67481739816X**3, 427819787963 + 894960934550X + 739173522309X**2 + 97557368455X**3



In [351]:
# add the coefficients of the 2 ciphertexts
def add(c1, c2, q):
    c0_add = add_poly_exact(c1[0], c2[0], q)
    c1_add = add_poly_exact(c1[1], c2[1], q)
    return (c0_add, c1_add)

(c_add0, c_add1) = add(enc_m1, enc_m2, q)
ciphertext = (c_add0, c_add1)
plaintext = decrypt_poly(ciphertext, secret_key, polynomial_modulus, q)

print(f"Encrypted sum: {pr(c_add0)}, {pr(c_add1)}")
print(f"-------------------------")
print(f"Expected sum: {pr(add_poly_exact(m1, m2, q))}")
print(f"Final decryption result: {pr(plaintext)}")

Encrypted sum: 281468700641 + 369811030444X + 806649657308X**2 + 437071530330X**3, 953196944378 + 262590453373X + 583386110064X**2 + 555452842384X**3
-------------------------
Expected sum: 368640 + 196138X + 418542X**2 + 103662X**3
Final decryption result: 368623 + 196130X + 418544X**2 + 103663X**3


We can see that the decyption result after the ciphetext addition is an approximation of the original sum (addition of the original polynomial messages).

## Multiplication


Multiplication in CKKS is more complex compared to addition because it introduces significant growth in both the ciphertext dimension and noise. After multiplying two ciphertexts, the resulting ciphertext will have three components (dimension 3), compared to the original ciphertext dimension of 2. This growth in dimension occurs because the homomorphic multiplication generates additional terms.

Additionally, the scale and noise grow faster during multiplication. For example, given two ciphertexts $c$ and $c'$ with scale $\Delta$, the resulting ciphertext will have terms scaled by $\Delta^2$. to address these issues, two kep operations are performed after multiplication: rescaling and relinearization.

$$ CMult(c,c') = c_{mult} = (d_0, d_1, d_2) = (c_0 \cdot c'_0, c_0 \cdot c'_1 + c'_0 \cdot c_1, c_1 \cdot c'_1)$$

### Rescaling
Rescaling is used to manage the growth of the scale $\Delta$ and reduce the noise after multiplication. Without rescaling, the scale would grow exponentially with successive multiplications, making decryption and further operations impractical.

Ciphertexts are initialized at a level L, and for every multiplication it decreases the level of the resulting ciphertext. For any binary operation to happen, the two involved ciphertext should have the same level, else the operation will fail. The modulo q can be defined as the following : $q = \Delta^L * q_0$ with $\Delta$ as the scaling factor and $q_0$ as the base modulus. With this the rescaling operation can be defined as : 
$$ RS_{l \rightarrow l-1}(c) = \lfloor \frac{q_{l-1}}{q_l}c \rceil (mod \space q_{l-1}) = \lfloor \Delta ^{-1}c \rceil (mod \space q_{l-1})$$

### Relinearization
Relinearization is used to manage the growth of the ciphertext dimension. After multiplication, the ciphertext dimension increases to 3, which makes further operations computationally expensive and memory-intensive. Relinearization reduces the ciphertext dimension back to 2, while preserving the correctness of the encrypted data.

To relinearize a 3-dimensional ciphertext,we use an evaluation key evk
$$ evk = (-a_0 \cdot s + e + p \cdot s^2, a)$$
$$ Relin((d_0, d_1, d_2), evk) = (d_0, d_1) + \lfloor p^{-1}*d_2*evk \rceil$$

We reuse the same ideas and formulas from the previous encryption section: generate a secret key, derive a public key $(b, a)$, encrypt a message as a pair $(c_0, c_1)$, and decrypt a multiplied ciphertext with $d_0 + d_1 s + d_2 s^2$.

The only part we do **not** reuse directly is the float-based `Polynomial` arithmetic, because that is exactly what breaks ciphertext multiplication for large moduli. In this section, we keep the tutorial flow the same but represent polynomials by exact integer coefficient lists.

In [352]:
# We now reuse the same exact helper functions that were introduced in the
# encryption and decryption section above. The multiplication step only adds
# the 3-component ciphertext formula and the corresponding decryption rule.

def multiply(c1, c2, polynomial_modulus, q):
    d0 = mul_poly_exact(c1[0], c2[0], q)
    d1 = add_poly_exact(mul_poly_exact(c1[0], c2[1], q), mul_poly_exact(c1[1], c2[0], q), q)
    d2 = mul_poly_exact(c1[1], c2[1], q)
    return (d0, d1, d2)

def decrypt_multiplied_poly(ciphertext, secret_key, polynomial_modulus, q):
    d0, d1, d2 = ciphertext
    s_squared = mul_poly_exact(secret_key, secret_key, q)
    partial = add_poly_exact(d0, mul_poly_exact(d1, secret_key, q), q)
    plaintext = add_poly_exact(partial, mul_poly_exact(d2, s_squared, q), q)
    return plaintext

In [353]:
# We use a zero-noise example here so that the multiplication formula can be checked exactly.
# The ciphertexts are still built from the same generate_keys / encrypt_poly flow as above,
# but we fix the randomness and set the noise polynomials to zero to inspect the algebra.

N = 16
h = 2
q = 2**31 - 1
polynomial_modulus = polynomial_from_dict({0: 1, N: 1})

m1 = pad_coefficients([25, 14, 25, 7], N)
m2 = pad_coefficients([15, 14, 35, 8], N)

# Two fixed ternary polynomials play the role of the encryption randomness.
u1 = pad_coefficients([1, 0, -1, 1], N)
u2 = pad_coefficients([0, 1, 1, -1], N)
zero_error = [0] * N

secret_key, public_key = generate_keys(N, q, h, polynomial_modulus, mu=0, Sigma=0, e=zero_error, seed=0)
enc_m1 = encrypt_poly(m1, N, public_key, polynomial_modulus, q, mu=0, Sigma=0, e1=zero_error, e2=zero_error, v=u1)
enc_m2 = encrypt_poly(m2, N, public_key, polynomial_modulus, q, mu=0, Sigma=0, e1=zero_error, e2=zero_error, v=u2)

d0, d1, d2 = multiply(enc_m1, enc_m2, polynomial_modulus, q)
decrypted = decrypt_multiplied_poly((d0, d1, d2), secret_key, polynomial_modulus, q)
expected = mul_poly_exact(m1, m2, q)

expected_centered = center_coefficients(expected, q)
decrypted_centered = center_coefficients(decrypted, q)

print(f"Message 1: {pr(m1[:8])}")
print(f"Message 2: {pr(m2[:8])}")
print(f"Ciphertext 1, c0: {pr(center_coefficients(enc_m1[0], q)[:8])}")

print(f"Ciphertext 1, c1: {pr(center_coefficients(enc_m1[1], q)[:8])}")
print(f"Ciphertext 2, c0: {pr(center_coefficients(enc_m2[0], q)[:8])}")
print(f"Ciphertext 2, c1: {pr(center_coefficients(enc_m2[1], q)[:8])}")
print("-------------------------")
print(f"d0: {pr(center_coefficients(d0, q)[:8])}")
print(f"d1: {pr(center_coefficients(d1, q)[:8])}")
print(f"d2: {pr(center_coefficients(d2, q)[:8])}")
print("-------------------------")
print(f"Expected product: {pr(expected_centered[:8])}")
print(f"Decrypted product: {pr(decrypted_centered[:8])}")
print(f"Ciphertext multiplication correct: {expected_centered == decrypted_centered}")

Message 1: 25 + 14X + 25X**2 + 7X**3
Message 2: 15 + 14X + 35X**2 + 8X**3
Ciphertext 1, c0: 956205909 + 511987405X - 59481316X**2 + 479536825X**3 + 33706898X**4 + 149506030X**5 - 96973628X**6 - 224388121X**7
Ciphertext 1, c1: 614403227 - 1040307263X + 470998513X**2 - 857985730X**3 - 84101379X**4 + 982985904X**5 - 917467071X**6 + 395435439X**7
Ciphertext 2, c0: 355352571 + 143746793X - 613223511X**2 - 715305603X**3 - 848902239X**4 + 470642992X**5 - 429920518X**6 - 809112912X**7
Ciphertext 2, c1: - 648331207 - 498759397X + 851997363X**2 - 1034140167X**3 - 143406841X**4 + 995744142X**5 + 500119934X**6 - 858712452X**7
-------------------------
d0: - 253659986 + 93960072X - 106142157X**2 - 877884368X**3 + 1019310895X**4 - 420510905X**5 - 801486308X**6 - 875853009X**7
d1: - 706668525 + 714359377X + 72531001X**2 + 877283927X**3 - 829313628X**4 - 846645188X**5 - 836365535X**6 + 904414174X**7
d2: 914560677 - 15951135X - 927984282X**2 - 165854214X**3 + 124795653X**4 - 591180862X**5 + 977870624X*